# Proyecto de Machine Learning
## Predicción del precio de seguros para mascotas

# Notebook 03 – Evaluación

Este notebook recoge la **evaluación de los modelos** entrenados en el Notebook 02. Su objetivo es medir el rendimiento de cada modelo mediante métricas de regresión, compararlos de forma homogénea, analizar la interpretabilidad (qué variables influyen más en el precio) y seleccionar el modelo final.

> **Estado:** estructura preparada. La evaluación se ejecutará cuando el modelado (Notebook 02) esté disponible. En esta fase **no se realizan evaluaciones todavía**.

# 1. Importación de librerías

Librerías necesarias para la carga de modelos/datos, el cálculo de métricas y la visualización de resultados.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)

import joblib  # para cargar / guardar modelos (pickle o joblib)

# 2. Configuración del entorno

Configuración de visualización para mantener un formato consistente durante la evaluación.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
np.set_printoptions(suppress=True)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# 3. Carga de modelos y conjunto de test

## Objetivo

Cargar los modelos entrenados en el Notebook 02 y el conjunto de **test** generado durante el preprocesamiento (partición 80/20 acordada por el equipo).

> ⚠️ **Pendiente de disponibilidad del modelado.** Esta sección se completará cuando Hugo entregue los modelos entrenados y las predicciones sobre el test. Los nombres/rutas son orientativos y se ajustarán a los que defina el Notebook 02.

In [ ]:
# --- PLACEHOLDER: descomentar y ajustar cuando el modelado esté disponible ---

# X_test = pd.read_pickle("../data/X_test.pkl")
# y_test = pd.read_pickle("../data/y_test.pkl")

# Diccionario {nombre_modelo: modelo_entrenado}
# modelos = {
#     "Baseline (media)": joblib.load("../models/baseline.pkl"),
#     "Regresión Lineal": joblib.load("../models/linear_regression.pkl"),
#     "Ridge":            joblib.load("../models/ridge.pkl"),
#     "Lasso":            joblib.load("../models/lasso.pkl"),
#     "Elastic Net":      joblib.load("../models/elastic_net.pkl"),
#     "Random Forest":    joblib.load("../models/random_forest.pkl"),
#     # "Gradient Boosting": joblib.load("../models/gradient_boosting.pkl"),
# }

# 4. Funciones de métricas

## Objetivo

Definir funciones reutilizables para calcular las métricas acordadas por el equipo: **MAE, RMSE, R² y MAPE**.

| Métrica | Qué mide | Lectura para negocio |
|---------|----------|----------------------|
| **MAE** | Error medio absoluto (en €) | Cuánto nos equivocamos de media, en euros |
| **RMSE** | Error cuadrático medio (en €) | Penaliza más los errores grandes |
| **R²** | Varianza explicada (0–1) | Qué parte de la variación del precio explica el modelo |
| **MAPE** | Error porcentual medio | Cuánto nos equivocamos de media, en % |

In [ ]:
def calcular_metricas(y_true, y_pred):
    """Devuelve un diccionario con MAE, RMSE, R2 y MAPE para una predicción.

    Parámetros
    ----------
    y_true : array-like  -> valores reales del precio_mensual
    y_pred : array-like  -> valores predichos por el modelo

    Devuelve
    --------
    dict con las cuatro métricas.
    """
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100  # en %

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "MAPE (%)": mape}

# 5. Tabla comparativa de modelos

## Objetivo

Recoger las métricas de todos los modelos en una única tabla para compararlos de forma homogénea. Se rellenará automáticamente al recorrer los modelos cargados en la sección 3.

In [ ]:
def construir_tabla_comparativa(modelos, X_test, y_test):
    """Recorre los modelos, calcula sus métricas sobre el test y devuelve
    un DataFrame comparativo ordenado por MAE (menor es mejor)."""
    filas = []
    for nombre, modelo in modelos.items():
        y_pred = modelo.predict(X_test)
        met = calcular_metricas(y_test, y_pred)
        met["Modelo"] = nombre
        filas.append(met)

    tabla = (
        pd.DataFrame(filas)
        .set_index("Modelo")
        .sort_values("MAE")
        .round(3)
    )
    return tabla

In [ ]:
# --- Se ejecutará cuando existan los modelos ---
# tabla_resultados = construir_tabla_comparativa(modelos, X_test, y_test)
# tabla_resultados

# Estructura de la tabla (vacía) como referencia:
tabla_resultados = pd.DataFrame(
    columns=["MAE", "RMSE", "R2", "MAPE (%)"]
)
tabla_resultados.index.name = "Modelo"
tabla_resultados

# 6. Visualizaciones

## Objetivo

Apoyar la comparación y la interpretación de resultados con gráficos. Cada función queda definida y lista para usarse cuando existan predicciones reales.

## 6.1 Comparativa de métricas entre modelos

In [ ]:
def graficar_comparativa_metricas(tabla, metrica="MAE"):
    """Gráfico de barras comparando una métrica entre todos los modelos."""
    orden = tabla[metrica].sort_values(ascending=(metrica != "R2"))
    ax = orden.plot(kind="barh")
    ax.set_title(f"Comparativa de modelos — {metrica}")
    ax.set_xlabel(metrica)
    plt.tight_layout()
    plt.show()

# graficar_comparativa_metricas(tabla_resultados, "MAE")

## 6.2 Valores reales vs. predichos

Permite ver visualmente cómo de cerca quedan las predicciones de la realidad (la nube ideal se pega a la diagonal).

In [ ]:
def graficar_real_vs_predicho(y_true, y_pred, titulo="Real vs. Predicho"):
    plt.scatter(y_true, y_pred, alpha=0.3)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims, "r--", label="Predicción perfecta")
    plt.xlabel("Precio real")
    plt.ylabel("Precio predicho")
    plt.title(titulo)
    plt.legend()
    plt.tight_layout()
    plt.show()

# graficar_real_vs_predicho(y_test, mejor_modelo.predict(X_test))

## 6.3 Distribución de residuos

Los residuos (real − predicho) deberían repartirse alrededor de 0 sin patrones claros.

In [ ]:
def graficar_residuos(y_true, y_pred):
    residuos = y_true - y_pred
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(residuos, kde=True, ax=axes[0])
    axes[0].set_title("Distribución de residuos")
    axes[0].set_xlabel("Residuo (real - predicho)")
    axes[1].scatter(y_pred, residuos, alpha=0.3)
    axes[1].axhline(0, color="r", linestyle="--")
    axes[1].set_title("Residuos vs. predicho")
    axes[1].set_xlabel("Precio predicho")
    axes[1].set_ylabel("Residuo")
    plt.tight_layout()
    plt.show()

# graficar_residuos(y_test, mejor_modelo.predict(X_test))

## 6.4 Importancia de variables / coeficientes

Responde a la pregunta del negocio: **¿qué variables influyen más en el precio?** Para modelos lineales se usan los coeficientes; para modelos de árbol, la importancia de variables.

In [ ]:
def graficar_importancia(modelo, nombres_variables, top=15):
    """Muestra la importancia de variables (árboles) o los coeficientes (lineales)."""
    if hasattr(modelo, "feature_importances_"):
        valores = modelo.feature_importances_
        titulo = "Importancia de variables"
    elif hasattr(modelo, "coef_"):
        valores = np.abs(modelo.coef_)
        titulo = "Coeficientes (valor absoluto)"
    else:
        print("El modelo no expone importancia de variables ni coeficientes.")
        return

    imp = (
        pd.Series(valores, index=nombres_variables)
        .sort_values(ascending=False)
        .head(top)
    )
    imp.sort_values().plot(kind="barh")
    plt.title(titulo)
    plt.tight_layout()
    plt.show()

# graficar_importancia(mejor_modelo, X_test.columns)

# 7. Revisión de overfitting (train vs. test)

## Objetivo

Comparar el rendimiento en **train** y en **test** de cada modelo. Una diferencia grande a favor de train indica sobreajuste (overfitting).

In [ ]:
def comparar_train_test(modelos, X_train, y_train, X_test, y_test, metrica="R2"):
    filas = []
    for nombre, modelo in modelos.items():
        m_train = calcular_metricas(y_train, modelo.predict(X_train))[metrica]
        m_test  = calcular_metricas(y_test,  modelo.predict(X_test))[metrica]
        filas.append({"Modelo": nombre, f"{metrica}_train": m_train,
                      f"{metrica}_test": m_test, "Diferencia": m_train - m_test})
    return pd.DataFrame(filas).set_index("Modelo").round(3)

# comparar_train_test(modelos, X_train, y_train, X_test, y_test)

# 8. Selección del modelo final y persistencia

## Objetivo

Seleccionar el modelo con mejor equilibrio entre **capacidad predictiva** e **interpretabilidad** (criterio acordado por el equipo) y guardarlo en `src/models` en formato `joblib`/`pickle`, tal como exige la guía del proyecto.

> El modelo definitivo se elige en equipo el 17 de julio. Esta celda deja preparada la persistencia.

In [ ]:
# --- Se ejecutará tras elegir el modelo final ---
# mejor_modelo = modelos["<nombre del modelo elegido>"]
# joblib.dump(mejor_modelo, "../models/modelo_final.pkl")
# print("Modelo final guardado en src/models/modelo_final.pkl")

# 9. Conclusiones

En esta fase la evaluación aún no se ha ejecutado (a la espera del Notebook 02). Cuando estén los modelos, esta sección recogerá:

- La tabla comparativa final de métricas.
- El modelo seleccionado y su justificación (predicción + interpretabilidad).
- Las variables más influyentes en el precio del seguro.
- Las conclusiones ligadas al objetivo de negocio y posibles acciones de mejora.